# Tutorial 13-4: The Mask of the Decoder
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

---
## 1. Bidirectional vs. Unidirectional Context
In the **Transformer Encoder** (like BERT), every word can see every other word in the sentence. This is called 'Bidirectional Context'.

However, in the **Transformer Decoder** (like GPT), we use the model to generate text one word at a time. During training, we can't let the model 'cheat' by looking at the word it is supposed to be predicting. 

**The Solution:** We apply a mask to the attention scores before the softmax step. This mask sets the scores of all future words to negative infinity ($-\infty$), which the softmax function turns into a probability of 0.

In [ ]:
%pip install transformers

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import BertTokenizer, BertModel

# Load a real model and tokenizer to get realistic attention scores
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

sentence = "The chef who made the pizza was happy."
inputs = tokenizer(sentence, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
seq_len = len(tokens)

with torch.no_grad():
    # Get the raw Query and Key projections from the first layer
    # We'll use these to calculate a realistic 'unmasked' attention matrix
    embeddings = model.embeddings(inputs['input_ids'])
    self_attn = model.encoder.layer[0].attention.self
    Q = self_attn.query(embeddings)
    K = self_attn.key(embeddings)
    
    # Calculate Raw Scores (QK^T / sqrt(dk))
    d_k = Q.shape[-1]
    raw_scores = torch.matmul(Q, K.transpose(-1, -2)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    # Remove batch dimension for visualization
    raw_scores = raw_scores.squeeze(0)

## 2. Creating the Causal Mask
We create a 'triangular' mask where the lower half (including the diagonal) is 0 and the upper half is $-\infty$.

In [ ]:
# Create an upper triangular matrix of ones
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

# Apply the mask: set upper triangle to -inf
masked_scores = raw_scores.masked_fill(mask, float('-inf'))

# Compute Softmax for both to see the distribution of attention
attn_unmasked = F.softmax(raw_scores, dim=-1)
attn_masked = F.softmax(masked_scores, dim=-1)

## 3. Visualizing the Difference
Notice how in the **Encoder View**, the word 'chef' can attend to 'pizza' (future). In the **Decoder View**, the attention to any word to the right is exactly zero.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 7))

# Plot Unmasked (Encoder Style)
im0 = ax[0].imshow(attn_unmasked.numpy(), cmap='viridis')
ax[0].set_title("Encoder Style (Bidirectional)\nEvery word can see the entire sentence")
ax[0].set_xticks(range(seq_len))
ax[0].set_xticklabels(tokens, rotation=45)
ax[0].set_yticks(range(seq_len))
ax[0].set_yticklabels(tokens)

# Plot Masked (Decoder Style)
im1 = ax[1].imshow(attn_masked.numpy(), cmap='viridis')
ax[1].set_title("Decoder Style (Unidirectional/Masked)\nWords can only see the past (left side)")
ax[1].set_xticks(range(seq_len))
ax[1].set_xticklabels(tokens, rotation=45)
ax[1].set_yticks(range(seq_len))
ax[1].set_yticklabels(tokens)

plt.tight_layout()
plt.show()

## 4. Summary
* **Causal Masking** is what turns a generic Transformer block into a **Decoder**.
* By setting future scores to $-\infty$, we ensure the model learns to predict the next word based solely on what has come before.
* This allows us to train the model on entire sentences in parallel, while still maintaining the autoregressive property (predicting the future from the past).